# Contenedor A:
##  Script copiado del main.py


In [2]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from autotec.scrapers import   scraper_dani, scraper_neiel, scraper_martin, scraper_belenandrades1, scraper_belenandrades3, scraper_belenandrades4, scraper_javiera, scraper_jocelyn, scraper_luz, scraper_martin2, scraper_belenandrades5

# Configuración de Spark (Mantenlo fuera de las funciones para no recrear la sesión)

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
spark = (
    SparkSession.builder
    .appName("AutoTec_Batch_Processing")
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

def procesar_y_guardar(lista_scrapers):
    """Ejecuta scrapers y realiza un UPSERT en MongoDB para evitar duplicados"""
    for nombre, funcion in lista_scrapers:
        print(f"\n🚀 Procesando: {nombre}")
        try:
            datos = funcion()
            if datos and len(datos) > 0:
                df = spark.createDataFrame(datos)

                df.write \
                    .format("mongodb") \
                    .mode("append") \
                    .option("database", "proyecto_bigdata") \
                    .option("collection", "raw_data") \
                    .option("operationType", "update") \
                    .option("upsertDocument", "true") \
                    .option("idFieldList", "url") \
                    .save()
                
                print(f"✅ {nombre}: {len(datos)} registros procesados (actualizados/insertados) en MongoDB.")
            else:
                print(f"⚠️ {nombre}: No se obtuvieron datos.")
        except Exception as e:
            print(f"❌ Error en {nombre}: {e}")
# --- DEFINICIÓN DE TANDAS ---

 #Tanda 1
grupo_1 = [
   ("Dani", scraper_dani.ejecutar_extraccion),
    ("Neiel", scraper_neiel.ejecutar_extraccion),
    ("Martin", scraper_martin.ejecutar_extraccion),
    ("Belen1", scraper_belenandrades1.ejecutar_extraccion),
    ("Belen3", scraper_belenandrades3.ejecutar_extraccion)
]

# Tanda 2
#grupo_2 = [
#    ("Luz", scraper_luz.ejecutar_extraccion),
#    ("Martin2", scraper_martin2.ejecutar_extraccion),
#    ("Belen4", scraper_belenandrades4.ejecutar_extraccion),
#    ("Jocelyn", scraper_jocelyn.ejecutar_extraccion),
#    ("Javiera", scraper_javiera.ejecutar_extraccion),
#    ("Belen5", scraper_belenandrades5.ejecutar_extraccion)

#]

# --- EJECUCIÓN ---
# Puedes comentar una línea para ejecutar solo la otra
print("Iniciando Tanda 1...")
procesar_y_guardar(grupo_1)

#print("Iniciando Tanda 2...")
#procesar_y_guardar(grupo_2)
print("\n¡Proceso de carga parcial completado!")
spark.stop()



¡Proceso de carga parcial completado!
